# Hệ thống chấm bài trắc nghiệm

Quy trình: in phiếu → học sinh tô bài → giáo viên chụp ảnh → upload đáp án và ảnh → nhận điểm.

**Cách chạy trên Colab:** chọn *Runtime → Run all*, rồi kéo xuống cuối notebook để mở giao diện. Phiếu trắng và dữ liệu mẫu nằm trong thanh *Files* bên trái.

In [ ]:
%pip install -q gradio==6.31.0

## Bước 1. Bố cục phiếu
Mọi tọa độ nằm ở một chỗ, dùng chung khi vẽ và khi đọc phiếu. Phiếu A4 ở 200 DPI có kích thước 1654×2339 px. Nhờ vậy, sau khi nắn ảnh về đúng kích thước này, ta biết chính xác vị trí từng ô.

In [ ]:
"""Toa do cua phieu tra loi, dung chung cho viec ve phieu va doc phieu."""

PAGE_W = 1654  # A4 o 200 DPI
PAGE_H = 2339
NUM_QUESTIONS = 40
CHOICES = "ABCD"
ID_DIGITS = 6
BUBBLE_RADIUS = 20
MARKER_SIZE = 60
MARKER_CENTERS = [(80, 80), (1574, 80), (1574, 2259), (80, 2259)]  # TL, TR, BR, BL
ORIENTATION_BAR = (677, 70, 300, 20)  # x, y, w, h: chi co o mep tren

_ID_X, _ID_Y, _ID_DX, _ID_DY = 1050, 260, 70, 62
_Q_X, _Q_Y, _Q_DX, _Q_DY = 250, 1000, 70, 110
_Q_COL_GAP = 350
_Q_PER_COL = 10


def question_bubble(q, choice):
    """Tam o tron cua cau q (1..40), lua chon choice (A..D)."""
    col, row = divmod(q - 1, _Q_PER_COL)
    return (_Q_X + col * _Q_COL_GAP + CHOICES.index(choice) * _Q_DX, _Q_Y + row * _Q_DY)


def id_bubble(pos, digit):
    """Tam o tron cua chu so thu pos (0..5) trong ma hoc sinh, gia tri digit (0..9)."""
    return (_ID_X + pos * _ID_DX, _ID_Y + digit * _ID_DY)

## Bước 2. Vẽ phiếu và xuất PDF
Phiếu có 4 ô vuông đen ở góc để định vị và một thanh đen chỉ ở mép trên. Bốn ô góc giống nhau nên không phân biệt được ảnh ngược đầu; thanh đen giải quyết việc đó.

In [ ]:
"""Ve phieu tra loi va xuat PDF de in."""
import unicodedata
from functools import lru_cache

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont


_FONTS = ["DejaVuSans.ttf", "LiberationSans-Regular.ttf", "Arial.ttf", "Arial Unicode.ttf"]


@lru_cache(maxsize=None)
def _load_font(size):
    """Tra ve (font, co_dau): may khong co font tieng Viet thi in khong dau."""
    for name in _FONTS:
        try:
            return ImageFont.truetype(name, size), True
        except OSError:
            continue
    return ImageFont.load_default(size), False


def _ascii(text):
    text = text.replace("Đ", "D").replace("đ", "d")
    return unicodedata.normalize("NFD", text).encode("ascii", "ignore").decode()


def render_sheet():
    """Ve phieu trang, tra ve anh xam uint8 kich thuoc PAGE_H x PAGE_W."""
    img = np.full((PAGE_H, PAGE_W), 255, np.uint8)
    half = MARKER_SIZE // 2
    for x, y in MARKER_CENTERS:
        img[y - half:y + half, x - half:x + half] = 0
    bx, by, bw, bh = ORIENTATION_BAR
    img[by:by + bh, bx:bx + bw] = 0

    pts = [question_bubble(q, c) for q in range(1, NUM_QUESTIONS + 1) for c in CHOICES]
    pts += [id_bubble(p, d) for p in range(ID_DIGITS) for d in range(10)]
    for pt in pts:
        cv2.circle(img, pt, BUBBLE_RADIUS, 70, 2, cv2.LINE_AA)

    pil = Image.fromarray(img)
    draw = ImageDraw.Draw(pil)

    def text(xy, s, size=26, anchor="mm"):
        font, has_vn = _load_font(size)
        draw.text(xy, s if has_vn else _ascii(s), font=font, fill=0, anchor=anchor)

    text((180, 170), "PHIẾU TRẢ LỜI TRẮC NGHIỆM", 40, "lm")
    text((180, 300), "Họ và tên: ____________________________", 30, "lm")
    text((180, 400), "Lớp: ______________", 30, "lm")
    text((180, 520), "Tô kín một ô cho mỗi câu, dùng bút chì 2B", 26, "lm")
    text((180, 565), "hoặc bút mực tối màu.", 26, "lm")
    text((id_bubble(0, 0)[0] + 175, 195), "MÃ HỌC SINH", 30)
    for d in range(10):
        text((id_bubble(0, d)[0] - 50, id_bubble(0, d)[1]), str(d))
    for q in range(1, NUM_QUESTIONS + 1):
        x, y = question_bubble(q, "A")
        text((x - 55, y), str(q))
        if (q - 1) % 10 == 0:
            for c in CHOICES:
                cx, cy = question_bubble(q, c)
                text((cx, cy - 50), c)
    return np.array(pil)


def save_pdf(path):
    Image.fromarray(render_sheet()).save(path, "PDF", resolution=200)

## Bước 3. Tìm phiếu và nắn ảnh
1. **Ảnh xám** và **lọc Gauss** để giảm nhiễu.
2. **Phân ngưỡng thích nghi**: mỗi điểm được so với vùng lân cận, nên ánh sáng không đều vẫn tách được nét đen.
3. **Tìm contour**: giữ các hình 4 đỉnh gần vuông, đặc và tối hơn hẳn giấy bao quanh, đó là ứng viên ô định vị. Bộ 4 ô đúng phải xếp đúng tỉ lệ phiếu và khớp với thanh định hướng, nhờ vậy loại được vật đen vuông ngoài phiếu.
4. **Biến đổi phối cảnh**: từ 4 cặp điểm tính ma trận 3×3, đưa ảnh nghiêng về khung chuẩn.

In [ ]:
"""Tim phieu trong anh chup va nan ve khung chuan."""
from itertools import combinations

import cv2
import numpy as np
from PIL import Image, ImageOps


MAX_SIDE = 3000
MAX_CANDIDATES = 12
MAX_INK_RATIO = 0.3  # o dinh vi la muc den in tren giay trang: trong toi hon han vien quanh
BAR_SLACK = 40  # px: giay cong lam thanh dinh huong lech khoi vi tri chuan sau khi nan
_MARKER_RATIO = (MARKER_CENTERS[2][1] - MARKER_CENTERS[1][1]) / (MARKER_CENTERS[1][0] - MARKER_CENTERS[0][0])


class SheetUnreadable(Exception):
    def __init__(self, reason):
        super().__init__(reason)
        self.reason = reason


def load_image(path):
    """Doc anh, ap dung huong EXIF cua dien thoai, gioi han kich thuoc; tra ve BGR."""
    try:
        with Image.open(path) as im:
            rgb = np.array(ImageOps.exif_transpose(im).convert("RGB"))
        scale = MAX_SIDE / max(rgb.shape[:2])
        if scale < 1:
            rgb = cv2.resize(rgb, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    except Exception as e:
        raise SheetUnreadable("Không đọc được file ảnh") from e
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def preprocess(bgr):
    """Anh xam -> loc nhieu -> phan nguong thich nghi (block lon hon duong kinh o tron)."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 101, 10)
    return gray, blurred, binary


def _ink_on_paper(gray, contour, area):
    """Trong contour toi hon han dai giay bao quanh; loai phim ban phim, vat toi tren nen toi.

    Chi xet khung bao cua contour noi rong them nua cua so gian no, khong tao mat na ca anh.
    """
    k = max(int(np.sqrt(area) * 0.25), 3) | 1
    x, y, w, h = cv2.boundingRect(contour)
    x0, y0 = max(x - k // 2, 0), max(y - k // 2, 0)
    roi = gray[y0:y + h + k // 2, x0:x + w + k // 2]
    mask = np.zeros(roi.shape, np.uint8)
    cv2.drawContours(mask, [contour], -1, 255, -1, offset=(-x0, -y0))
    ring = cv2.dilate(mask, np.ones((k, k), np.uint8)) - mask
    return np.median(roi[mask > 0]) < MAX_INK_RATIO * np.median(roi[ring > 0])


def _candidates(binary, gray):
    """Contour gan vuong, dac, lon, la muc tren giay: ung vien o dinh vi. Tra ve [(dien_tich, tam)]."""
    contours, _ = cv2.findContours(binary, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    min_area = (max(binary.shape) / 200) ** 2
    found = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area or area > 0.02 * binary.size:
            continue
        approx = cv2.approxPolyDP(c, 0.06 * cv2.arcLength(c, True), True)
        if len(approx) != 4 or not cv2.isContourConvex(approx):
            continue
        (cx, cy), (w, h), _ = cv2.minAreaRect(c)
        # phoi canh manh lam o vuong thanh hinh binh hanh, nen do dac tinh theo bao loi
        if min(w, h) == 0 or not 0.5 < w / h < 2.0 or area / cv2.contourArea(cv2.convexHull(c)) < 0.9:
            continue
        if not _ink_on_paper(gray, c, area):
            continue
        # vien trong va vien ngoai cua cung mot o cho hai contour trung tam: chi giu mot
        if all(np.hypot(cx - fx, cy - fy) > 10 for _, (fx, fy) in found):
            found.append((area, (cx, cy)))
    return sorted(found, key=lambda f: -f[0])[:MAX_CANDIDATES]


def _order(pts):
    """Sap xep 4 goc theo vong, canh dai cua tu giac la chieu doc cua phieu."""
    center = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    p = pts[np.argsort(angles)]
    if np.linalg.norm(p[0] - p[1]) > np.linalg.norm(p[1] - p[2]):
        p = np.roll(p, -1, axis=0)
    return p


def _rect_error(p):
    """Do lech cua tu giac so voi hinh chu nhat ti le nhu phieu; None neu khong hop le."""
    if not cv2.isContourConvex(p.astype(np.float32)):
        return None
    top, right, bottom, left = (np.linalg.norm(p[(i + 1) % 4] - p[i]) for i in range(4))
    if min(top, bottom) < 100:
        return None
    if max(top, bottom) / min(top, bottom) > 1.5 or max(left, right) / min(left, right) > 1.5:
        return None
    ratio_err = abs((left + right) / (top + bottom) - _MARKER_RATIO) / _MARKER_RATIO
    return ratio_err if ratio_err < 0.35 else None


_BAR_X0 = ORIENTATION_BAR[0] + 20  # vung chua thanh dinh huong trong khung chuan, cho lech +-BAR_SLACK
_BAR_Y0 = ORIENTATION_BAR[1] - BAR_SLACK
_BAR_SIZE = (ORIENTATION_BAR[2] - 40, ORIENTATION_BAR[3] + 2 * BAR_SLACK)  # (rong, cao)
# toa do khung chuan cua vung o mep tren va (sau khi xoay 180 do) o mep duoi
_BAR_TOP = np.float64([[1, 0, _BAR_X0], [0, 1, _BAR_Y0], [0, 0, 1]])
_BAR_BOTTOM = np.float64([[-1, 0, PAGE_W - 1 - _BAR_X0], [0, -1, PAGE_H - 1 - _BAR_Y0], [0, 0, 1]])


def _bar_level(band, darkest):
    """Muc sang cua dai cao bang thanh dinh huong, toi nhat (hoac sang nhat) trong vung."""
    h = ORIENTATION_BAR[3] - 8
    bands = np.convolve(band.mean(axis=1), np.ones(h) / h, mode="valid")
    return bands.max() if darkest else bands.min()


def _bar_band(img):
    w, h = _BAR_SIZE
    return img[_BAR_Y0:_BAR_Y0 + h, _BAR_X0:_BAR_X0 + w]


def _bar_present(binary, corners):
    """Dung bo 4 o thi thanh dinh huong phai nam o mep tren hoac duoi.

    Chi nan vung quanh thanh dinh huong (khong nan ca phieu) vi buoc nay chay cho moi bo 4 o.
    """
    m = cv2.getPerspectiveTransform(np.float32(MARKER_CENTERS), np.float32(corners)).astype(np.float64)
    top, bottom = (_bar_level(cv2.warpPerspective(binary, m @ place, _BAR_SIZE,
                                                  flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP), True)
                   for place in (_BAR_TOP, _BAR_BOTTOM))
    return max(top, bottom) > 0.7 * 255 and min(top, bottom) < 0.3 * 255


def _outside(sheet, other):
    """Bo 4 o nay nam han ngoai phieu da chon: trong anh co phieu thu hai."""
    return all(cv2.pointPolygonTest(sheet, (float(x), float(y)), False) < 0 for x, y in other)


def find_markers(binary, gray):
    """Tim 4 o dinh vi: to hop phai dung ti le phieu va khop thanh dinh huong."""
    ranked = []
    for combo in combinations(_candidates(binary, gray), 4):
        areas = [c[0] for c in combo]
        if max(areas) / min(areas) > 2.0:
            continue
        p = _order(np.float32([c[1] for c in combo]))
        err = _rect_error(p)
        if err is not None:
            ranked.append((err, p))
    valid = [p for _, p in ranked if _bar_present(binary, p)]
    if not valid:
        raise SheetUnreadable("Không tìm thấy 4 ô định vị")
    # bo lon nhat la phieu that; cac o da to ben trong phieu cung co the xep thanh bo gia
    valid.sort(key=lambda p: -np.linalg.norm(p[0] - p[2]))
    if any(_outside(valid[0], other) for other in valid[1:]):
        raise SheetUnreadable("Ảnh có nhiều hơn một phiếu")
    return valid[0]


def warp(gray, corners):
    """Bien doi phoi canh ve khung chuan PAGE_W x PAGE_H."""
    m = cv2.getPerspectiveTransform(np.float32(corners), np.float32(MARKER_CENTERS))
    return cv2.warpPerspective(gray, m, (PAGE_W, PAGE_H), flags=cv2.INTER_LINEAR)


def fix_orientation(warped):
    """Dua vao thanh den chi co o mep tren de xac dinh chieu phieu, xoay 180 do neu nguoc."""
    top = _bar_level(_bar_band(warped), False)
    flipped = cv2.rotate(warped, cv2.ROTATE_180)
    bottom = _bar_level(_bar_band(flipped), False)
    if min(top, bottom) > 0.6 * max(top, bottom):
        raise SheetUnreadable("Không xác định được chiều phiếu")
    return warped if top < bottom else flipped

## Bước 4. Đọc ô tô
- **Bù giấy cong**: tìm viền các ô in, ghép với vị trí chuẩn, ước lượng độ lệch mềm (đa thức rồi nội suy Gauss cục bộ) và nắn lại ảnh.
- **Khử ánh sáng**: phép đóng hình thái ước lượng nền giấy, chia ảnh cho nền và cho mức sáng giấy của chính phiếu để giấy gần 1 và mực gần 0.
- **Căn lại tâm ô**: khớp mẫu vòng tròn với viền in để bù sai số còn lại sau khi nắn.
- **Tỉ lệ điểm đen** trong lõi mỗi ô (bỏ viền in): dưới 0.25 là trống, trên 0.75 là đã tô, ở giữa là không chắc. Ô tô đủ diện tích nhưng quá nhạt cũng là không chắc.
- **"Trống" phải có bằng chứng**: phần trong viền ô và giấy quanh ô đều sạch. Nếu không, ô được coi là không chắc, nhờ vậy nét tô nhạt, vết tẩy và bóng đổ không bị chấm nhầm.
- **Kiểm tra chất lượng**: độ dốc tại cạnh ô định vị cho biết ảnh mờ; không thấy viền in của nhiều ô nghĩa là ảnh lóa hoặc lưới lệch.

In [ ]:
"""Doc cac o da to tren phieu da nan va kiem tra chat luong anh."""
from dataclasses import dataclass, field

import cv2
import numpy as np


BLANK_BELOW = 0.25  # ti le diem den trong o
FILLED_ABOVE = 0.75
DARK_PIXEL = 0.75  # do sang so voi nen giay; toi hon muc nay la diem den
SOLID_MEAN = 0.65  # o da to phai du dam; to nhat la "khong chac"
# "O trong" duoc so voi o trong dien hinh cua chinh phieu (trung vi cua moi o): do mo, vien in day,
# nhieu hat giay cua anh chup co san trong mau nen tu triet tieu. Gia tri la muc toi hon mau.
INNER_DEV = 0.08  # trung binh phan trong vien: vet tay do, net to nhat
OUTSIDE_DEV = 0.07  # giay quanh o: bong do, vat den
INK_DEV = 0.25  # diem muc dam
MAX_INNER_INK = 0.02  # net to nho, vien o to bi loa
FAINT_DEV = 0.07  # diem hoi toi (sau khi lam min de bo nhieu hat): net chi nhat
MAX_INNER_FAINT = 0.08  # o trong that toi da ~0.06, net nhat rat mo tren anh mo ~0.10
MAX_DARKNESS_GAP = 0.15  # hai o cung to phai dam nhu nhau, khac nhieu la vet tay
MAX_MISSING_RINGS = 0.05
MIN_SHARPNESS = 0.07  # mo hon muc nay thi net to nhat co the bi nhoe mat; mo nhe van doc dung
RING_COVERAGE = 0.7
RING_CONTRAST = 0.5  # vien thay duoc khi con it nhat ngan nay do tuong phan cua vien o mau
STEP_MAX_SIDE = 1000


@dataclass
class SheetReading:
    student_id: str | None = None
    id_problems: list = field(default_factory=list)
    marks: dict = field(default_factory=dict)
    uncertain: list = field(default_factory=list)
    problems: list = field(default_factory=list)
    steps: dict = field(default_factory=dict)
    warped: np.ndarray | None = None


def classify_ratio(ratio):
    if ratio < BLANK_BELOW:
        return "blank"
    return "filled" if ratio > FILLED_ABOVE else "uncertain"


def _normalize_light(warped):
    """Loc Gauss roi chia cho nen giay uoc luong duoc de khu anh sang khong deu; giay ~ 1, muc ~ 0."""
    blurred = cv2.GaussianBlur(warped, (5, 5), 0)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (61, 61))
    background = cv2.morphologyEx(blurred, cv2.MORPH_CLOSE, kernel)
    return blurred.astype(np.float32) / np.maximum(background.astype(np.float32), 1)


_R = int(BUBBLE_RADIUS * 0.7)
_P = BUBBLE_RADIUS + 10
_YY, _XX = np.mgrid[-_P:_P + 1, -_P:_P + 1]
_DIST = np.hypot(_XX, _YY)
_CORE = _DIST <= _R
_INNER = _DIST <= BUBBLE_RADIUS - 5  # chua 1 px cho cho giay cong lam vien o nho lai
_FAINT_ZONE = _DIST <= BUBBLE_RADIUS - 7  # net nhat chi xet xa vien, vien lech nhe khong lam nham
_OUTSIDE = (_DIST >= BUBBLE_RADIUS + 4) & (_DIST <= _P)
_ANGLES = np.arange(36) * 2 * np.pi / 36


def _rays(radii):
    """Toa do lay mau trong patch theo 36 huong. Nhieu diem roi dung vao x.5: lam tron len co
    chu dong (+1e-9) de sai so bit cuoi khong doi diem lay mau, giong het ban MATLAB."""
    pts = [[(_P + r * np.sin(a), _P + r * np.cos(a)) for r in radii] for a in _ANGLES]
    return np.floor(np.array(pts) + 0.5 + 1e-9).astype(int)


# vung vien (R-4..R+4) va giay ngay ben ngoai (ben trong co the da to kin)
_RAYS_RING = _rays(range(BUBBLE_RADIUS - 4, BUBBLE_RADIUS + 5))
_RAYS_PAPER = _rays((BUBBLE_RADIUS + 8, BUBBLE_RADIUS + 9))


_SNAP = 12  # px: sai so nan cuc bo toi da duoc bu lai; nho hon nhieu so voi khoang cach hai o
_T = BUBBLE_RADIUS + 9


def _ring_template():
    """Vong duong o vien in, vong am ngay ben ngoai: o da to cung chi khop dung o mot vi tri.

    Hai vong deu chuan hoa tong = 1, nen diem khop = do toi cua vien tru do toi cua giay quanh.
    """
    yy, xx = np.mgrid[-_T:_T + 1, -_T:_T + 1]
    dist = np.hypot(xx, yy)
    pos = (np.abs(dist - BUBBLE_RADIUS) <= 2.5).astype(np.float32)  # vong day 5 px
    neg = (np.abs(dist - BUBBLE_RADIUS - 6) <= 2.5).astype(np.float32)
    return pos / pos.sum() - neg / neg.sum()


_RING_TEMPLATE = _ring_template()


def ring_score(light):
    """Ban do diem khop vien o tren anh da khu sang (khong qua nhi phan nen chiu duoc anh mo).

    score[y - _T, x - _T] la diem cua o co tam (x, y).
    """
    return cv2.matchTemplate(1 - light, _RING_TEMPLATE, cv2.TM_CCORR)


def _snap(score, center):
    """Can lai tam o theo vien in gan nhat, bu sai so phoi canh va giay cong."""
    x, y = center
    window = score[y - _T - _SNAP:y - _T + _SNAP + 1, x - _T - _SNAP:x - _T + _SNAP + 1]
    _, best, _, (dx, dy) = cv2.minMaxLoc(window)
    return (x, y) if best <= 0 else (x + dx - _SNAP, y + dy - _SNAP)


_PEAK_COUNT = 2  # giu so dinh manh nhat gap doi so o; dinh thua (chu, vet ban) bi loai khi ghep
_POLY_STEPS = ((1, 45), (2, 25), (3, 15))  # (bac da thuc, ban kinh ghep o) tu tho den min
_LOCAL_STEPS = (30, 20, 12, 8)  # ban kinh ghep o khi bu cong cuc bo (vung gan tay cam)
_LOCAL_SIGMA = 60  # px: tam anh huong cua mot o len do lech cua vung quanh no
_MIN_MATCHED = 0.5
_MAX_BEND = 80  # px: lech lon hon muc nay thi khong tin la giay cong
_ANCHOR_WEIGHT = 5


def _ring_peaks(score):
    """Tam cac vien o tim thay tren toan phieu da nan, manh nhat truoc."""
    peaks = (score >= cv2.dilate(score, np.ones((21, 21), np.uint8))) & (score > 0)
    ys, xs = np.nonzero(peaks)
    keep = np.argsort(-score[ys, xs])[:_PEAK_COUNT * len(_all_bubbles())]
    return np.column_stack([xs[keep] + _T, ys[keep] + _T]).astype(np.float32)


def _poly(pts, deg):
    u, v = pts[:, 0] / PAGE_W, pts[:, 1] / PAGE_H
    return np.column_stack([u ** i * v ** j for i in range(deg + 1) for j in range(deg + 1 - i)])


def _smooth(src, values, pts):
    """Noi suy Gauss cac gia tri tai src ve pts; xa moi o thi tat dan ve 0."""
    d2 = ((pts[:, None] - src[None]) ** 2).sum(axis=2)
    w = np.exp(-d2 / (2 * _LOCAL_SIGMA ** 2))
    return (w @ values) / (w.sum(axis=1) + 0.05)[:, None]


def _bend_field(score):
    """Uoc luong do lech giua vi tri o chuan va vi tri o that; None neu khong tin cay.

    Tra ve ham tinh do lech tai cac diem bat ky. Ghep o chuan voi vien o gan nhat, tu ban kinh
    lon va da thuc bac thap toi bac cao, roi bu phan du cuc bo cho cho giay cong gat.
    O dinh vi da duoc nan dung nen co do lech 0.
    """
    template = np.float32(_all_bubbles())
    peaks = _ring_peaks(score)
    if len(peaks) < 2:
        return None

    def match(pred, radius):
        dist = np.linalg.norm(pred[:, None] - peaks[None], axis=2)
        order = np.argsort(dist, axis=1)[:, :2]
        d1, d2 = np.take_along_axis(dist, order, axis=1).T
        ok = (d1 < radius) & (d2 > 1.5 * d1)
        return ok, peaks[order[:, 0]] - template

    anchors = np.repeat(np.float32(MARKER_CENTERS), _ANCHOR_WEIGHT, axis=0)
    disp = np.zeros_like(template)
    for deg, radius in _POLY_STEPS:
        ok, target = match(template + disp, radius)
        if ok.mean() < _MIN_MATCHED:
            return None
        src = np.vstack([template[ok], anchors])
        dst = np.vstack([target[ok], np.zeros_like(anchors)])
        coef = np.linalg.lstsq(_poly(src, deg), dst, rcond=None)[0]
        disp = _poly(template, deg) @ coef

    def base(pts):
        return _poly(pts, deg) @ coef

    for radius in _LOCAL_STEPS:
        ok, target = match(template + disp, radius)
        if ok.mean() < _MIN_MATCHED:
            return None
        src, residual = template[ok], target[ok] - base(template[ok])
        disp = base(template) + _smooth(src, residual, template)
    return lambda pts: base(pts) + _smooth(src, residual, pts)


_GRID_STEP = 16  # px: tinh do lech tren luoi thua roi noi suy tuyen tinh ra tung diem anh


def _lerp_matrix(n, count):
    """Ma tran noi suy tuyen tinh tu luoi 0, step, 2*step, ... (count diem) ra cac toa do 0..n-1."""
    pos = np.arange(n) / _GRID_STEP
    i = np.minimum(pos.astype(int), count - 2)
    t = pos - i
    m = np.zeros((n, count), np.float32)
    m[np.arange(n), i] = 1 - t
    m[np.arange(n), i + 1] = t
    return m


def unbend(warped):
    """Bu giay cong: nan lai anh de cac o that trung voi vi tri chuan; khong tin cay thi giu nguyen."""
    field = _bend_field(ring_score(_normalize_light(warped)))
    if field is None:
        return warped
    gy, gx = np.mgrid[0:PAGE_H + _GRID_STEP:_GRID_STEP, 0:PAGE_W + _GRID_STEP:_GRID_STEP]
    disp = field(np.column_stack([gx.ravel(), gy.ravel()]).astype(np.float32))
    if np.abs(disp).max() > _MAX_BEND:
        return warped
    my, mx = _lerp_matrix(PAGE_H, gx.shape[0]), _lerp_matrix(PAGE_W, gx.shape[1])
    dx = my @ disp[:, 0].reshape(gx.shape).astype(np.float32) @ mx.T
    dy = my @ disp[:, 1].reshape(gx.shape).astype(np.float32) @ mx.T
    xs, ys = np.meshgrid(np.arange(PAGE_W, dtype=np.float32), np.arange(PAGE_H, dtype=np.float32))
    return cv2.remap(warped, xs + dx, ys + dy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def _patch(norm, center):
    x, y = center
    return norm[y - _P:y + _P + 1, x - _P:x + _P + 1]


def _bubble_state(norm, center, blank_ref):
    """Ty le diem den trong loi o -> filled / blank / uncertain.

    "Trong" can bang chung: ben trong vien in va giay xung quanh giong o trong dien hinh cua phieu.
    """
    patch = _patch(norm, center)
    core = patch[_CORE]
    dev = blank_ref - patch
    if dev[_OUTSIDE].mean() > OUTSIDE_DEV:
        return "uncertain"
    state = classify_ratio(float((core < DARK_PIXEL).mean()))
    if state == "filled":
        return "filled" if core.mean() <= SOLID_MEAN else "uncertain"
    inner = dev[_INNER]
    faint = cv2.blur(dev.astype(np.float32), (7, 7))[_FAINT_ZONE]
    if (inner.mean() > INNER_DEV or (inner > INK_DEV).mean() > MAX_INNER_INK
            or (faint > FAINT_DEV).mean() > MAX_INNER_FAINT):
        return "uncertain"
    return state


def _blank_reference(norm, score):
    """O trong dien hinh cua phieu: trung vi patch cua moi o (da so o la o trong).

    Chia ca anh va mau cho muc giay cua mau de giay ~ 1 bat ke giay xam hay anh toi.
    """
    ref = np.median([_patch(norm, _snap(score, p)) for p in _all_bubbles()], axis=0)
    level = max(float(ref[_OUTSIDE].mean()), 0.5)
    return norm / level, ref / level


def _ring_contrast(patch):
    """Do tuong phan vien in theo 36 huong: giay ben ngoai tru diem toi nhat trong vung vien."""
    ring = patch[_RAYS_RING[..., 0], _RAYS_RING[..., 1]].min(axis=1)
    paper = patch[_RAYS_PAPER[..., 0], _RAYS_PAPER[..., 1]].mean(axis=1)
    return paper - ring


def _ring_visible(norm, center, blank_ref):
    """Vien in cua o co ro nhu o trong dien hinh khong? Bi loa che thi mat vien o nhieu huong."""
    seen = _ring_contrast(_patch(norm, center)) >= RING_CONTRAST * _ring_contrast(blank_ref)
    return seen.mean() >= RING_COVERAGE


def _all_bubbles():
    pts = [question_bubble(q, c) for q in range(1, NUM_QUESTIONS + 1) for c in CHOICES]
    return pts + [id_bubble(p, d) for p in range(ID_DIGITS) for d in range(10)]


def _edge_sharpness(warped):
    """Do doc lon nhat tai canh trai cua cac o dinh vi, chia cho do tuong phan; canh net ~ 0.5."""
    half = MARKER_SIZE // 2
    values = []
    for x, y in MARKER_CENTERS:
        profile = warped[y - 15:y + 15, x - half - 25:x - half + 25].mean(axis=0)
        contrast = float(profile.max() - profile.min())
        values.append(float(np.abs(np.diff(profile)).max()) / max(contrast, 1.0))
    return min(values)


def check_quality(warped, norm, score, blank_ref):
    """Bao anh mo qua do net cua canh; bao loa/lech khi khong thay vien cua nhieu o."""
    problems = []
    if _edge_sharpness(warped) < MIN_SHARPNESS:
        problems.append("Ảnh bị mờ")
    pts = _all_bubbles()
    missing = sum(not _ring_visible(norm, _snap(score, p), blank_ref) for p in pts)
    if missing / len(pts) > MAX_MISSING_RINGS:
        problems.append("Ảnh bị lóa hoặc lưới ô bị lệch")
    return problems


def _read_id(norm, score, blank_ref):
    digits, problems = [], []
    for pos in range(ID_DIGITS):
        states = [_state(norm, score, id_bubble(pos, d), blank_ref) for d in range(10)]
        filled = [d for d, s in enumerate(states) if s == "filled"]
        if "uncertain" in states or len(filled) != 1:
            problems.append(f"Mã học sinh: chữ số thứ {pos + 1} không đọc được")
        else:
            digits.append(str(filled[0]))
    return ("".join(digits) if not problems else None), problems


def _state(norm, score, center, blank_ref):
    """O trong ma khong thay vien (bi loa che) thi khong the coi la chac chan trong."""
    center = _snap(score, center)
    state = _bubble_state(norm, center, blank_ref)
    if state == "blank" and not _ring_visible(norm, center, blank_ref):
        return "uncertain"
    return state


def _uneven_marks(norm, score, q, filled):
    """Hai o to dam khac han nhau: o nhat hon co the la vet tay chua sach."""
    if len(filled) < 2:
        return False
    means = [_patch(norm, _snap(score, question_bubble(q, c)))[_CORE].mean() for c in filled]
    return max(means) - min(means) > MAX_DARKNESS_GAP


def _small(img):
    scale = STEP_MAX_SIDE / max(img.shape[:2])
    return cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else img


def read_sheet(path):
    """Doc mot anh bai lam; anh khong doc duoc thi ghi ly do vao problems, khong raise."""
    r = SheetReading()
    try:
        bgr = load_image(path)
        r.steps["goc"] = _small(bgr)
        gray, _, binary = preprocess(bgr)
        r.steps["xam"] = _small(gray)
        r.steps["nhi_phan"] = _small(binary)
        corners = find_markers(binary, gray)
        marked = bgr.copy()
        for x, y in corners:
            cv2.circle(marked, (int(x), int(y)), 40, (0, 0, 255), 8)
        r.steps["dinh_vi"] = _small(marked)
        warped = fix_orientation(warp(gray, corners))
    except SheetUnreadable as e:
        r.problems.append(e.reason)
        return r
    r.steps["nan_phoi_canh"] = _small(warped)
    warped = unbend(warped)
    r.warped = warped
    r.steps["nan_giay_cong"] = _small(warped)
    light = _normalize_light(warped)
    score = ring_score(light)
    norm, blank_ref = _blank_reference(light, score)
    r.steps["khu_anh_sang"] = _small((np.clip(norm, 0, 1) * 255).astype(np.uint8))
    r.problems += check_quality(warped, norm, score, blank_ref)
    r.student_id, r.id_problems = _read_id(norm, score, blank_ref)
    for q in range(1, NUM_QUESTIONS + 1):
        states = {c: _state(norm, score, question_bubble(q, c), blank_ref) for c in CHOICES}
        r.marks[q] = [c for c, s in states.items() if s == "filled"]
        if "uncertain" in states.values() or _uneven_marks(norm, score, q, r.marks[q]):
            r.uncertain.append(q)
    return r

## Bước 5. Chấm điểm
Điểm = số câu đúng / N × 10. Câu trống hoặc tô nhiều ô được 0 điểm và có ghi chú. Bài có điểm không chắc chắn được đánh dấu **Cần xem lại** và không cho điểm: thà để giáo viên xem lại còn hơn chấm sai.

In [ ]:
"""Doc dap an, cham diem, ve anh danh dau va xuat CSV."""
import csv
import os
from collections import Counter
from dataclasses import dataclass, field

import cv2
import numpy as np


DONE = "Đã chấm"
REVIEW = "Cần xem lại"
_GREEN, _RED = (0, 170, 0), (0, 0, 230)


class AnswerKeyError(Exception):
    pass


@dataclass
class SheetResult:
    filename: str
    status: str
    student_id: str | None = None
    correct: int | None = None
    score: float | None = None
    notes: list = field(default_factory=list)
    annotated: np.ndarray | None = None
    steps: dict = field(default_factory=dict)


def _trim_row(row):
    """Bo khoang trang va cac cot rong o cuoi dong."""
    cells = [c.strip() for c in row]
    while cells and not cells[-1]:
        cells.pop()
    return cells


def load_answer_key(path):
    """Doc CSV hai cot cau,dap_an; chap nhan file luu tu Excel (BOM, dau ;, chu thuong)."""
    try:
        with open(path, encoding="utf-8-sig", newline="") as f:
            lines = [ln for ln in f.read().splitlines() if ln.strip()]
    except (OSError, UnicodeDecodeError) as e:
        raise AnswerKeyError("Không đọc được file đáp án") from e
    if lines and lines[0].lower().startswith("sep="):
        lines = lines[1:]
    if not lines:
        raise AnswerKeyError("File đáp án trống")
    delim = ";" if ";" in lines[0] else ","
    rows = [_trim_row(row) for row in csv.reader(lines, delimiter=delim)]
    rows = [row for row in rows if row]  # Excel hay de lai dong toan dau phan cach
    if not rows or [c.lower() for c in rows[0]] != ["cau", "dap_an"]:
        raise AnswerKeyError("Dòng đầu của file đáp án phải là: cau,dap_an")
    key = {}
    for num, row in enumerate(rows[1:], start=2):
        if len(row) != 2 or not row[0].isdigit():
            raise AnswerKeyError(f"Dòng {num} sai định dạng")
        q, answer = int(row[0]), row[1].upper()
        if len(answer) != 1 or answer not in CHOICES:
            raise AnswerKeyError(f"Dòng {num}: đáp án phải là một trong A, B, C, D")
        if q != len(key) + 1:
            raise AnswerKeyError(f"Dòng {num}: số câu phải liên tục từ 1")
        key[q] = answer
    if not 1 <= len(key) <= NUM_QUESTIONS:
        raise AnswerKeyError(f"Đáp án phải có từ 1 đến {NUM_QUESTIONS} câu")
    return key


def _annotate(warped, marks, key):
    """Vong xanh o dap an dung, vong do o hoc sinh to sai."""
    out = cv2.cvtColor(warped, cv2.COLOR_GRAY2BGR)
    for q, answer in key.items():
        cv2.circle(out, question_bubble(q, answer), BUBBLE_RADIUS + 6, _GREEN, 4)
        for c in marks.get(q, []):
            if c != answer:
                cv2.circle(out, question_bubble(q, c), BUBBLE_RADIUS + 6, _RED, 4)
    return out


def grade_reading(filename, reading, key):
    """Bai doc khong chac chan thi "Can xem lai", khong bao gio cho diem."""
    result = SheetResult(filename, DONE, reading.student_id, steps=reading.steps)
    uncertain = [q for q in reading.uncertain if q in key]
    notes = list(reading.problems) + list(reading.id_problems)
    if uncertain:
        notes.append("Câu không đọc chắc chắn: " + ", ".join(map(str, uncertain)))
    if reading.warped is not None:
        result.annotated = _annotate(reading.warped, reading.marks, key)
    if notes:
        result.status, result.notes = REVIEW, notes
        return result
    blank = [q for q in key if not reading.marks.get(q)]
    multi = [q for q in key if len(reading.marks.get(q, [])) > 1]
    if blank:
        notes.append("Câu bỏ trống: " + ", ".join(map(str, blank)))
    if multi:
        notes.append("Câu tô nhiều ô: " + ", ".join(map(str, multi)))
    result.correct = sum(reading.marks.get(q) == [a] for q, a in key.items())
    result.score = round(result.correct / len(key) * 10, 2)
    result.notes = notes
    return result


def grade_batch(paths, key):
    results = []
    for p in paths:
        name = os.path.basename(p)
        try:
            results.append(grade_reading(name, read_sheet(p), key))
        except Exception as e:  # mot file loi khong duoc lam mat ket qua cua ca lop
            results.append(SheetResult(name, REVIEW, notes=[f"Lỗi khi xử lý ảnh: {e}"]))
    counts = Counter(r.student_id for r in results if r.student_id)
    for r in results:
        if counts.get(r.student_id, 0) > 1:
            r.notes.append(f"Trùng mã học sinh {r.student_id}")
    return results


def write_csv(results, path):
    """utf-8-sig de Excel hien dung tieng Viet."""
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(["ten_file", "ma_hoc_sinh", "so_cau_dung", "diem", "trang_thai", "ghi_chu"])
        for r in results:
            w.writerow([r.filename, r.student_id or "",
                        "" if r.correct is None else r.correct,
                        "" if r.score is None else r.score,
                        r.status, "; ".join(r.notes)])

## Bước 6. Sinh ảnh mô phỏng
Tạo ảnh có xoay, nghiêng, tối và ánh sáng không đều với đáp án biết trước, dùng để thử hệ thống khi chưa có phiếu thật.

In [ ]:
"""Sinh anh mo phong anh chup phieu, da biet truoc dap an."""
import math

import cv2
import numpy as np



def fill_bubble(img, center, darkness=40, coverage=1.0):
    """To mot o; coverage < 1 chi to phan tren cua o, mo phong net to thieu."""
    x, y = center
    r = int(BUBBLE_RADIUS * 0.9)
    mask = np.zeros(img.shape, np.uint8)
    cv2.circle(mask, (x, y), r, 255, -1)
    mask[y - r + int(2 * r * coverage):] = 0
    img[mask > 0] = darkness


def fill_sheet(img, student_id, answers):
    """Tra ve ban sao cua phieu da to ma hoc sinh va cac cau tra loi."""
    out = img.copy()
    for pos, ch in enumerate(student_id):
        fill_bubble(out, id_bubble(pos, int(ch)))
    for q, choice in answers.items():
        fill_bubble(out, question_bubble(q, choice))
    return out


def photograph(img, angle=0.0, perspective=0.0, brightness=1.0, gradient=0.0, blur=0,
               background=200, seed=0):
    """Mo phong anh chup: dat phieu len nen, xoay, nghieng, doi sang, lam mo, them nhieu."""
    rng = np.random.default_rng(seed)
    side = int(math.hypot(PAGE_W, PAGE_H)) + 200
    cx, cy = side / 2, side / 2
    a = math.radians(angle)
    corners = []
    for sx, sy in [(-1, -1), (1, -1), (1, 1), (-1, 1)]:
        dx = sx * PAGE_W / 2 + rng.uniform(-perspective, perspective) * PAGE_W
        dy = sy * PAGE_H / 2 + rng.uniform(-perspective, perspective) * PAGE_W
        corners.append((cx + dx * math.cos(a) - dy * math.sin(a),
                        cy + dx * math.sin(a) + dy * math.cos(a)))
    src = np.float32([[0, 0], [PAGE_W, 0], [PAGE_W, PAGE_H], [0, PAGE_H]])
    m = cv2.getPerspectiveTransform(src, np.float32(corners))
    out = cv2.warpPerspective(img, m, (side, side), flags=cv2.INTER_LINEAR,
                              borderValue=int(background)).astype(np.float32)
    out *= brightness
    if gradient:
        out *= np.linspace(1.0, 1.0 - gradient, side, dtype=np.float32)[None, :]
    if blur:
        k = blur | 1
        out = cv2.GaussianBlur(out, (k, k), 0)
    out += rng.normal(0, 2.5, out.shape).astype(np.float32)
    return cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)


def add_glare(img):
    """Phu mot vung trang loa len khu vuc o tra loi cua phieu xam."""
    out = img.copy()
    x, y = question_bubble(11, "A")
    cv2.circle(out, (x + 100, y + 300), 320, 255, -1)
    return out


def bend(img, dx=30, dy=40, center=(250, 1000), sigma=300):
    """Lam cong giay quanh mot diem (nhu cho tay cam): noi dung lech toi (dx, dy), o goc gan nhu khong lech."""
    ys, xs = np.mgrid[0:img.shape[0], 0:img.shape[1]].astype(np.float32)
    w = np.exp(-((xs - center[0]) ** 2 + (ys - center[1]) ** 2) / (2 * sigma ** 2))
    return cv2.remap(img, xs - dx * w, ys - dy * w, cv2.INTER_LINEAR, borderValue=255)


def move_bar(img, dy=-22):
    """Dich thanh dinh huong theo chieu doc, mo phong giay cong o mep tren."""
    out = img.copy()
    x, y, w, h = ORIENTATION_BAR
    out[y:y + h, x:x + w] = 255
    out[y + dy:y + dy + h, x:x + w] = 0
    return out


def add_dark_keys(bgr, count=14, size=45, key=60, base=140):
    """Phu nen kieu ban phim: nhieu phim xam toi, lon hon o dinh vi, tren nen chi sang hon mot chut."""
    out = bgr.copy()
    out[:, :120] = base
    out[:120, :] = base
    for i in range(count):
        x = 10 + (i % 7) * 110
        out[20:20 + size, 120 + x:120 + x + size] = key
        out[200 + (i // 7) * 110:200 + (i // 7) * 110 + size, 20:20 + size] = key
    return out


def shift_content(img, dx=25):
    """Dich noi dung so voi o dinh vi, mo phong luoi o bi lech."""
    out = img.copy()
    box = out[170:2200, 160:1500].copy()
    out[170:2200, 160:1500] = 255
    out[170:2200, 160 + dx:1500 + dx] = box
    return out

## Bước 7. Giao diện Gradio

In [ ]:
"""Giao dien Gradio: upload dap an va bai lam, xem diem va tung buoc xu ly."""
import os
import tempfile

import cv2
import gradio as gr


COLUMNS = ["Tên file", "Mã học sinh", "Số câu đúng", "Điểm", "Trạng thái", "Ghi chú"]


def _rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img


def run_grading(key_path, image_paths):
    """Tra ve (hang cua bang, duong dan CSV, ket qua, thong bao)."""
    if not key_path:
        return [], None, [], "Hãy upload file đáp án."
    if not image_paths:
        return [], None, [], "Hãy upload ít nhất một ảnh bài làm."
    try:
        key = load_answer_key(key_path)
    except AnswerKeyError as e:
        return [], None, [], f"Lỗi file đáp án: {e}"
    results = grade_batch(image_paths, key)
    rows = [[r.filename, r.student_id or "", r.correct, r.score, r.status, "; ".join(r.notes)]
            for r in results]
    csv_path = os.path.join(tempfile.mkdtemp(), "scores.csv")
    write_csv(results, csv_path)
    review = sum(r.score is None for r in results)
    return rows, csv_path, results, f"Đã xử lý {len(results)} bài, {review} bài cần xem lại."


def show_sheet(index, results):
    """Tra ve (anh danh dau, [(anh tung buoc, ten buoc)]) cua bai thu index."""
    if index is None or not results or index >= len(results):
        return None, []
    r = results[index]
    annotated = None if r.annotated is None else _rgb(r.annotated)
    return annotated, [(_rgb(img), step) for step, img in r.steps.items()]


def build_ui():
    with gr.Blocks(title="Chấm bài trắc nghiệm") as demo:
        gr.Markdown("# Hệ thống chấm bài trắc nghiệm")
        results = gr.State([])
        with gr.Row():
            key = gr.File(label="1. File đáp án (CSV: cau,dap_an)", file_count="single",
                          file_types=[".csv"])
            images = gr.File(label="2. Ảnh bài làm (chọn được nhiều file)", file_count="multiple",
                             file_types=["image"])
        run = gr.Button("Chấm bài", variant="primary")
        message = gr.Markdown()
        table = gr.Dataframe(headers=COLUMNS, datatype="str", label="Kết quả", interactive=False)
        csv_file = gr.File(label="Tải file điểm (Excel có thể ẩn số 0 đầu mã học sinh)", interactive=False)
        choice = gr.Dropdown(label="Xem chi tiết bài", choices=[])
        with gr.Row():
            annotated = gr.Image(label="Đánh dấu: xanh là đáp án đúng, đỏ là ô tô sai")
            steps = gr.Gallery(label="Các bước xử lý ảnh", columns=3)

        def on_run(key_path, image_paths):
            rows, csv_path, res, msg = run_grading(key_path, image_paths)
            names = [(f"{i + 1}. {r.filename}", i) for i, r in enumerate(res)]
            return (rows, csv_path, res, msg, gr.update(choices=names, value=None), None, [])

        run.click(on_run, [key, images], [table, csv_file, results, message, choice, annotated, steps])
        choice.change(show_sheet, [choice, results], [annotated, steps])
    return demo

## Bước 8. Tạo phiếu, dữ liệu mẫu và thử một bài

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

Path("templates").mkdir(exist_ok=True)
save_pdf("templates/answer_sheet.pdf")

mau = Path("samples")
mau.mkdir(exist_ok=True)
dap_an = {q: "ABCD"[q % 4] for q in range(1, 41)}
(mau / "sample_answer_key.csv").write_text(
    "cau,dap_an\n" + "".join(f"{q},{a}\n" for q, a in dap_an.items()))

bai_lam = dict(dap_an)
bai_lam[3] = "A" if dap_an[3] != "A" else "B"  # co y to sai cau 3
del bai_lam[7]  # co y bo trong cau 7
phieu = fill_sheet(render_sheet(), "001234", bai_lam)
anh_mau = mau / "sample_submission.jpg"
cv2.imwrite(str(anh_mau), photograph(phieu, angle=6, perspective=.03, gradient=.4))

ket_qua = grade_batch([str(anh_mau)], load_answer_key(str(mau / "sample_answer_key.csv")))[0]
print(ket_qua.status, ket_qua.student_id, ket_qua.score, ket_qua.notes)

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for ax in axes.flat:
    ax.axis("off")
for ax, (ten, anh) in zip(axes.flat, ket_qua.steps.items()):
    ax.imshow(anh[..., ::-1] if anh.ndim == 3 else anh, cmap="gray")
    ax.set_title(ten)
plt.show()

plt.figure(figsize=(8, 11))
plt.imshow(ket_qua.annotated[..., ::-1])
plt.axis("off")
plt.show()

## Bước 9. Mở giao diện chấm bài

In [ ]:
import os

demo = build_ui()
if not os.environ.get("OMR_SKIP_LAUNCH"):
    # share=False: giao dien hien ngay trong Colab, khong tao link cong khai chua bai cua hoc sinh.
    # Neu khung giao dien khong hien, doi thanh share=True (bat ky ai co link deu vao duoc).
    demo.launch(share=False)